## Introduction

This notebook shows how to build a neural network with NNVM python frontend and generate runtime library for Raspberry Pi with TVM. To run this notebook, you need to install tvm and nnvm following [these instructions](https://github.com/dmlc/nnvm/blob/master/docs/how_to/install.md). Notice that you need to build tvm with llvm.

In [1]:
import tvm
import nnvm.compiler
import nnvm.testing

TVM: Initializing cython mode...


## Define Neural Network in nnvm

First, let's define a neural network with nnvm python frontend. For simplicity, we'll use pre-defined resnet-18 network in nnvm. Parameters are initialized with Xavier initializer. In this tutorial, we assume we will do inference on our device and the batch size is set to be 1. Input images are RGB color images of size 224 * 224.

In [2]:
batch_size = 1
image_shape = (3, 224, 224)

data_shape = (batch_size,) + image_shape

net, params = nnvm.testing.resnet.get_workload(batch_size=batch_size, image_shape=image_shape)
print(net.debug_str())

Symbol Outputs:
	output[0]=softmax(0)
Variable:data
Variable:bn_data_gamma
Variable:bn_data_beta
Variable:bn_data_moving_mean
Variable:bn_data_moving_var
--------------------
Op:batch_norm, Name=bn_data
Inputs:
	arg[0]=data(0) version=0
	arg[1]=bn_data_gamma(0) version=0
	arg[2]=bn_data_beta(0) version=0
	arg[3]=bn_data_moving_mean(0) version=0
	arg[4]=bn_data_moving_var(0) version=0
Attrs:
	epsilon=2e-05
Variable:conv0_weight
--------------------
Op:conv2d, Name=conv0
Inputs:
	arg[0]=bn_data(0)
	arg[1]=conv0_weight(0) version=0
Attrs:
	channels=64
	kernel_size=(7, 7)
	padding=(3, 3)
	strides=(2, 2)
	use_bias=False
Variable:bn0_gamma
Variable:bn0_beta
Variable:bn0_moving_mean
Variable:bn0_moving_var
--------------------
Op:batch_norm, Name=bn0
Inputs:
	arg[0]=conv0(0)
	arg[1]=bn0_gamma(0) version=0
	arg[2]=bn0_beta(0) version=0
	arg[3]=bn0_moving_mean(0) version=0
	arg[4]=bn0_moving_var(0) version=0
Attrs:
	epsilon=2e-05
--------------------
Op:relu, Name=relu0
Inputs:
	arg[0]=bn0(0)
-

## Compile with tvm

Next step is to compile model with tvm. Here we set optimization level to be 0 to indicate "SimplifyInference", which will minimize the size of built library.

In [3]:
opt_level = 0
with nnvm.compiler.build_config(opt_level=opt_level):
    graph, lib, params = nnvm.compiler.build(
        net, tvm.target.rasp(), shape={"data": data_shape}, params=params)

lib_fname = 'net.o'
lib.save(lib_fname)

A net.o library file will be created under the current folder with size 225KB.